# 17 · Partitions, shuffles, caching, lazy evaluation (Spark)

- A **partition** = one chunk of a DataFrame. One **task** processes one partition on one CPU core.
- **Narrow** transformation (`filter`, `select`, `withColumn`): each output partition needs only one input
  partition. No data moves.
- **Wide** transformation (`groupBy`, `join`, `distinct`, `orderBy`, `repartition`): rows must move between
  partitions = a **shuffle** (`Exchange` in the plan). Shuffles split a job into **stages**.
- **AQE** (Adaptive Query Execution) changes the plan while running: merges small shuffle partitions,
  switches to broadcast joins, splits skewed partitions.

This practice Spark uses `local[*]` (all your cores) and `spark.sql.shuffle.partitions = 4`.

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/17_partitions_shuffle.ipynb`](../../spark/practice/17_partitions_shuffle.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. How many partitions?  ·  🟢 easy

Print the number of partitions of: `orders` (read from a file), `orders.repartition(8)`, and `spark.range(0, 1000, numPartitions=6)`.

**How it works:** `df.rdd.getNumPartitions()` shows how many chunks (= tasks) the DataFrame has.

In [ ]:
print(orders.rdd.getNumPartitions())
print(orders.repartition(8).rdd.getNumPartitions())
print(spark.range(0, 1000, numPartitions=6).rdd.getNumPartitions())

1
8
6

**Why / traps:** For files, Spark makes about one partition per 128 MB (`spark.sql.files.maxPartitionBytes`). Our tiny file = 1 partition.

## Q2. repartition vs coalesce  ·  🟡 medium

Start from `spark.range(1000, numPartitions=8)`. Make 2 partitions with `coalesce(2)` and with `repartition(2)`. Print both plans: which one has an `Exchange` (shuffle)?

**How it works:** The plans show the difference: `coalesce` is a simple `Coalesce` step; `repartition` adds an `Exchange` (shuffle).

In [ ]:
df = spark.range(1000, numPartitions=8)
df.coalesce(2).explain()
df.repartition(2).explain()

== Physical Plan ==
Coalesce 2
+- *(1) Range (0, 1000, step=1, splits=8)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=9564]
   +- Range (0, 1000, step=1, splits=8)

**Why / traps:** `coalesce(n)` only **merges** partitions (no shuffle, can only go down; partitions may be uneven). `repartition(n)` does a full shuffle (can go up or down, even sizes). `repartition("col")` puts equal keys together.

## Q3. Rows per partition  ·  🟡 medium

Repartition `orders` by `customer_id` into 4 partitions and count rows per partition with `F.spark_partition_id()`.

**How it works:** `spark_partition_id()` tells which partition each row is in; groupBy it to see the row count per partition.

In [ ]:
res = (orders.repartition(4, "customer_id")
       .groupBy(F.spark_partition_id().alias("partition")).count()
       .orderBy("partition"))
res.show()

+---------+-----+
|partition|count|
+---------+-----+
|        0|   17|
|        1|   35|
|        2|   26|
|        3|   30|
+---------+-----+

**Why / traps:** Uneven counts = **skew**. Hash partitioning sends every row of one customer to the same partition.

## Q4. Narrow vs wide: find the shuffle  ·  🟢 easy

Print the plan of: orders → filter completed → withColumn year → groupBy year → count. Which step makes the `Exchange`?

**How it works:** `explain()` shows the plan; the `Exchange` line appears at the groupBy.

In [ ]:
(orders.filter("status = 'completed'")
       .withColumn("yr", F.year("order_date"))
       .groupBy("yr").count()
       .explain())

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[yr#1691], functions=[count(1)])
   +- Exchange hashpartitioning(yr#1691, 4), ENSURE_REQUIREMENTS, [plan_id=9768]
      +- HashAggregate(keys=[yr#1691], functions=[partial_count(1)])
         +- Project [year(order_date#14) AS yr#1691]
            +- Filter (isnotnull(status#16) AND (status#16 = completed))
               +- FileScan csv [order_date#14,status#16] Batched: false, DataFilters: [isnotnull(status#16), (status#16 = completed)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:<repo>/data/orders.tsv], PartitionFilters: [], PushedFilters: [IsNotNull(status), EqualTo(status,completed)], ReadSchema: struct<order_date:date,status:string>

**Why / traps:** filter and withColumn are narrow (same stage). groupBy is wide: `HashAggregate (partial)` → `Exchange hashpartitioning(yr)` → `HashAggregate (final)`. The partial step runs **before** the shuffle, so less data moves (like a combiner in MapReduce).

## Q5. Lazy evaluation  ·  🟢 easy

Build a DataFrame with a filter and a new column, and time it. Then call `count()` and time that. Which one actually did the work?

**How it works:** Building the DataFrame takes almost no time (just a plan); `count()` is when the work runs.

In [ ]:
import time
t0 = time.time()
df = spark.range(5_000_000).filter("id % 3 = 0").withColumn("sq", F.col("id") * F.col("id"))
print(f"transformations: {time.time() - t0:.3f}s")
t0 = time.time()
print(df.count())
print(f"action (count):  {time.time() - t0:.3f}s")

transformations: 0.010s
1666667
action (count):  0.177s

**Why / traps:** Transformations only build a plan (a **DAG**: a graph of steps). Spark runs nothing until an **action**. That lets the optimizer (Catalyst) rearrange the whole plan, e.g. push filters down.

## Q6. cache and persist  ·  🟡 medium

Make an expensive DataFrame (`spark.range(3_000_000)` with a groupBy), `cache()` it, run `count()` twice and time both. Check `is_cached`, then `unpersist()`.

**How it works:** `cache()` marks the DataFrame; the first count computes and stores it, the second reads from memory (faster).

In [ ]:
import time
df = spark.range(3_000_000).withColumn("k", F.col("id") % 1000).groupBy("k").agg(F.sum("id").alias("s"))
df.cache()
for i in (1, 2):
    t0 = time.time(); df.count(); print(f"run {i}: {time.time() - t0:.3f}s")
print("is_cached:", df.is_cached, "|", df.storageLevel)
df.unpersist()

run 1: 0.238s
run 2: 0.027s
is_cached: True | Disk Memory Deserialized 1x Replicated

**Why / traps:** `cache()` is lazy too: the first action fills it, later actions read from memory. Cache only data you reuse **several** times; it takes executor memory. `persist(StorageLevel.MEMORY_AND_DISK)` lets you choose the storage level.

## Q7. Shuffle partitions and AQE  ·  🟡 medium

Print `spark.sql.shuffle.partitions` and `spark.sql.adaptive.enabled`. Then run a groupBy on `orders` and print the output partition count with AQE **on** and **off**.

**How it works:** Read the settings with `spark.conf.get`; with AQE on, the groupBy output shrinks to fewer partitions.

In [ ]:
print(spark.conf.get("spark.sql.shuffle.partitions"), spark.conf.get("spark.sql.adaptive.enabled"))
for aqe in ("true", "false"):
    spark.conf.set("spark.sql.adaptive.enabled", aqe)
    n = orders.groupBy("status").count().rdd.getNumPartitions()
    print(f"AQE {aqe}: {n} partition(s) after groupBy")
spark.conf.set("spark.sql.adaptive.enabled", "true")

4 true
AQE true: 1 partition(s) after groupBy
AQE false: 4 partition(s) after groupBy

**Why / traps:** Without AQE you always get `shuffle.partitions` (default **200**: far too many for small data, too few for huge data). AQE **coalesces** small shuffle partitions after seeing the real sizes. A common rule of thumb: shuffle partitions ≈ 2–3 × total cores.

## Q8. Cut a long lineage with checkpoint  ·  🔴 hard

Run a loop that adds a column 30 times. Print the length of the plan text before and after `localCheckpoint()`.

**How it works:** A loop of withColumn makes a long plan; `localCheckpoint()` saves the data and replaces the plan with a short one.

In [ ]:
df = spark.range(100)
for i in range(30):
    df = df.withColumn(f"c{i}", F.col("id") + i)
plan_len = lambda d: len(d._jdf.queryExecution().logical().toString())
print("plan length before:", plan_len(df))
df = df.localCheckpoint()
print("plan length after: ", plan_len(df))

plan length before: 7349
plan length after:  302

**Why / traps:** **Lineage** = the full list of steps to rebuild a DataFrame. Loops (like the recursion in topic 09) make it huge and slow to plan. `checkpoint()` (to reliable storage) or `localCheckpoint()` (executor storage) saves the data and starts a fresh, short plan.

## Q9. Spark SQL hints for partitions  ·  🟢 easy

In Spark SQL, use the `REPARTITION(3)` and `COALESCE(1)` hints on `orders` and check the partition count.

**How it works:** SQL hints in `/*+ ... */` ask Spark to repartition or coalesce the query result.

In [ ]:
%%sql
SELECT /*+ REPARTITION(3) */ order_id, status FROM orders

,order_id,status
0,1017,completed
1,1099,cancelled
2,1049,completed
3,1043,completed
4,1104,cancelled
5,1065,completed
6,1014,completed
7,1064,completed
8,1006,completed
9,1037,completed


**DataFrame API:**

In [ ]:
for hint in ("REPARTITION(3)", "COALESCE(1)"):
    df = spark.sql(f"SELECT /*+ {hint} */ * FROM orders")
    print(hint, df.rdd.getNumPartitions())

REPARTITION(3) 3
COALESCE(1) 1

**Why / traps:** SQL users get the same controls as the DataFrame API: `REPARTITION`, `COALESCE`, `REPARTITION_BY_RANGE`, `BROADCAST`, `MERGE`, `SHUFFLE_HASH`.